In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

%pip install torchvision

In [2]:
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])


train_dataset = CIFAR10(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = CIFAR10(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))



Training samples: 50000
Testing samples: 10000


In [3]:
trainloader =DataLoader(train_dataset , batch_size = 64 , shuffle= True)
testloader = DataLoader(test_dataset , batch_size = 64)

### CNN

In [4]:

import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # Block 1: 3 -> 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 32 -> 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x


In [5]:
model = CNN()


In [6]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training our CNN

In [7]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    model.train()

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images)

        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch} & loss={epoch_training_loss / len(trainloader)}")
    
    


epoch=0 & loss=1.370804040023433
epoch=1 & loss=0.9554104734869564
epoch=2 & loss=0.7665412975546649
epoch=3 & loss=0.6367408881330734
epoch=4 & loss=0.5375672289172707
epoch=5 & loss=0.4411012895996004
epoch=6 & loss=0.36179244598311844
epoch=7 & loss=0.2842271481176167
epoch=8 & loss=0.22594734030249325
epoch=9 & loss=0.1784484729342296


In [8]:
# Evaluate our model
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)

        predicted = torch.max(outputs, 1).indices

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100:.2f}%")


Accuracy = 74.25%
